<a href="https://colab.research.google.com/github/marynatarasevych/ML_Study/blob/main/Tarasevych_%22HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import TomekLinks


pd.set_option('display.max.rows',130)
pd.set_option('display.max.columns',130)
pd.set_option('float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Hanna Pylieva/ДЗ/Модуль 2.2 Логістична регресія/Customer Segmentation Train.csv")

In [4]:
train.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.00,Low,4.00,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.00,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.00,Low,1.00,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.00,High,2.00,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.00,Cat_6,A


In [5]:
train.info(verbose = True, show_counts = True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [6]:
#null count
null_count = train.isnull().sum()
null_percentage = round((train.isnull().sum()/train.shape[0])*100, 2)
null_df = pd.DataFrame({'column_name' : train.columns,'null_count' : null_count,'null_percentage': null_percentage})
null_df.reset_index(drop = True, inplace = True)
null_df.sort_values(by = 'null_percentage', ascending = False)

,column_name,null_count,null_percentage
6,Work_Experience,829,10.28
8,Family_Size,335,4.15
2,Ever_Married,140,1.74
5,Profession,124,1.54
4,Graduated,78,0.97
9,Var_1,76,0.94
0,ID,0,0.00
1,Gender,0,0.00
3,Age,0,0.00
7,Spending_Score,0,0.00


In [7]:
# Створюємо трен. і вал. набори
train_df, val_df = train_test_split(train, test_size=0.20, random_state=42, stratify=train['Segmentation'])
cols_to_drop = ['ID', 'Segmentation']
input_cols = train_df.drop(columns=cols_to_drop).columns.tolist()
target_col = 'Segmentation'
train_inputs, train_targets = train_df[input_cols].copy(), train_df[target_col].copy()
val_inputs, val_targets = val_df[input_cols].copy(), val_df[target_col].copy()

# Виявляємо числові і категоріальні колонки
numeric_cols = train_inputs.select_dtypes(include=np.number).columns.tolist()
categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

# Створюємо трансформери для числових і категоріальних колонок
# Створюємо функцію для генерування нових фічей (Gender_Type та Age_Group)

def add_age_group(df):
    bins = [18, 25, 35, 45, 55, 65, 100]
    labels = ['18-25', '25-35', '35-45', '45-55', '55-65', '65+']
    df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
    return df

feature_engineering = Pipeline(steps=[
    ('age_group', FunctionTransformer(add_age_group)),
])

# Оновлюємо список колонок
categorical_cols = categorical_cols + ['Age_Group']

# Створюємо трансформери для числових і категоріальних колонок
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='if_binary'))
])

# Комбінуємо трансформери для різних типів колонок в один препроцесор
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])
# Стоврюємо пайплайн, який спочатку запускає фіча інженерінг і препроцесинг , потім тренуєм модель
model_pipeline = Pipeline(steps=[
    ('feature_engineering', feature_engineering),
    ('preprocessor', preprocessor),
    ('classifier', OneVsRestClassifier(LogisticRegression(solver='liblinear')))
])

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [8]:
# SMOTENC - трансформер без MinMaxScaler та OneHotEncoder

# Спочатку застосовуємо feature engineering, щоб додати Age_Group
train_inputs_fe = feature_engineering.fit_transform(train_inputs)

preprocessor_smotenc = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', SimpleImputer(strategy='most_frequent'), categorical_cols)
],
    verbose_feature_names_out=False  # прибираємо префікси num__/cat__
)

preprocessor_smotenc.set_output(transform='pandas')
train_inputs_smotenc = preprocessor_smotenc.fit_transform(train_inputs_fe)
#train_inputs_smotenc = pd.DataFrame(train_inputs_smotenc, columns=numeric_cols + categorical_cols)

# індекси категоріальних колонок у цьому датафреймі
cat_feature_indices = [train_inputs_smotenc.columns.get_loc(c) for c in categorical_cols]

smotenc = SMOTENC(categorical_features=cat_feature_indices, random_state=42)
X_train_smote_raw, y_train_smote = smotenc.fit_resample(train_inputs_smotenc, train_targets)

In [9]:
#SMOTE-Tomek
preprocessor_tomek = ColumnTransformer(transformers=[
    ('num', MinMaxScaler(), numeric_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='if_binary'), categorical_cols)
],
    verbose_feature_names_out=False  # прибираємо префікси num__/cat__
)

preprocessor_tomek.set_output(transform='pandas')
X_train_smote_tomek = preprocessor_tomek.fit_transform(X_train_smote_raw)

tomek = TomekLinks()
X_train_smotetomek_raw, y_train_smotetomek = tomek.fit_resample(X_train_smote_tomek, y_train_smote)

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [11]:
# Модель 1: Original
model_pipeline.fit(train_inputs, train_targets)

# Модель 2: SMOTE
model_smote = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model_smote.fit(X_train_smote_tomek, y_train_smote) # щоб до X_train_smote_raw були примінені MinMaxScaler та OneHotEncoder

# Модель 3: SMOTE-Tomek
model_smotetomek = OneVsRestClassifier(LogisticRegression(solver='liblinear'))
model_smotetomek.fit(X_train_smotetomek_raw, y_train_smotetomek)

OneVsRestClassifier(estimator=LogisticRegression(solver='liblinear'))

In [12]:
val_inputs_fe = feature_engineering.transform(val_inputs)  # додаємо Age_Group

val_inputs_imputed = preprocessor_smotenc.transform(val_inputs_fe)  # імпутація

val_inputs_encoded = preprocessor_tomek.transform(val_inputs_imputed)  # OneHot + MinMaxScaler

In [13]:
# Модель 1: Original
val_preds_original = model_pipeline.predict(val_inputs)
print("Original")
print(classification_report(val_targets, val_preds_original))

# Модель 2: SMOTE
val_preds_smote = model_smote.predict(val_inputs_encoded)
print("SMOTENC")
print(classification_report(val_targets, val_preds_smote))

# Модель 3: SMOTE-Tomek
val_preds_smotetomek = model_smotetomek.predict(val_inputs_encoded)
print("SMOTE-Tomek")
print(classification_report(val_targets, val_preds_smotetomek))

Original
              precision    recall  f1-score   support

           A       0.42      0.47      0.45       394
           B       0.42      0.18      0.25       372
           C       0.50      0.63      0.56       394
           D       0.63      0.74      0.68       454

    accuracy                           0.52      1614
   macro avg       0.50      0.50      0.48      1614
weighted avg       0.50      0.52      0.49      1614

SMOTENC
              precision    recall  f1-score   support

           A       0.42      0.49      0.45       394
           B       0.40      0.23      0.30       372
           C       0.52      0.60      0.55       394
           D       0.66      0.69      0.68       454

    accuracy                           0.52      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.51      0.52      0.50      1614

SMOTE-Tomek
              precision    recall  f1-score   support

           A       0.39      0.66      0.49   

**Висновки**

1. Напишіть, яку метрику ви обрали для порівняння моделей.

Для порівняння моделей я вибрала метрику `macro avg`, оскільки вона не враховує розмір кожного класу. Якщо враховувати, як `weighted avg`, то краща оцінка більш представленого класу перекрила б погані оцінки менших класів

2. Яка модель найкраща?

SMOTENC вийшла трошечки кращою за оригінальну модель і SMOTE-Tomek

3. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

Немає сильного дисбалансу класів - кажен займає приблизно 24% від всіх данних (тільки клас D трохи більше - 28%). Over_sampling та under_sampling працюють для збалансування класів, але тут вони і так збалансовані